![Noteable.ac.uk Banner](https://raw.githubusercontent.com/YusufNik/Noteable/refs/heads/main/images/Noteable%20NB%20Header%20Banner.png)

## Exemplar Information

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Purpose:</b> This exemplar introduces geospatial analysis in Python using real OpenStreetMap data for central Edinburgh. It covers importing and cleaning street, building, and green-space layers; projecting spatial data for metric analysis; creating regular analysis grids; calculating urban morphology indicators; comparing scale and threshold choices; visualising results with static and interactive maps; and interpreting hotspot-style spatial clustering responsibly.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Intended audience / teaching context:</b> Designed for students in introductory to intermediate Python, GIS, or urban geoscience teaching sessions. Suitable for classroom demonstration, guided lab work, or independent exploratory practice in spatial analysis and interactive mapping.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Noteable Requirements:</b><br>
    <b>Environment:</b> BETA 2026<br>
    <b>Server:</b> GeoScience<br>
    <b>Kernel:</b> Python 3
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Required data or dependencies:</b>
    <br>Real OpenStreetMap data downloaded at runtime for a central Edinburgh study area, including walking network, building footprints, and green-space polygons.<br>
    Python packages: <code>numpy</code>, <code>pandas</code>, <code>geopandas</code>, <code>osmnx</code>, <code>folium</code>, <code>plotly</code>, <code>matplotlib</code>, <code>ipywidgets</code>, <code>libpysal</code>, <code>esda</code><br>
    Standard library modules: <code>warnings</code>, <code>os</code>
</div>

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Date created / last reviewed:</b> 13 July 2026<br>
    <b>Maintainer / owner:</b> Nik Yusuf
</div>

# From Streets to Spatial Insight: An Interactive Edinburgh Urban Morphology Notebook

## Legend

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>blue</b>, the <b>instructions</b> and <b>goals</b> are highlighted. This tells you what we are trying to achieve.
</div>
<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>green</b>, key <b>information</b> and <b>concept explanations</b> are highlighted.
</div>
<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>yellow</b>, <b>exercises</b> and <b>tasks</b> are highlighted for you to try yourself.
</div>
<div style="border:1px solid #f5c6cb; background:#f8d7da; color:#721c24; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>red</b>, <b>error interpretation</b> and <b>debugging tips</b> are highlighted.
</div>

## 1. Setting Up Our Toolkit

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Import the geospatial tools we need. We will combine street-network analysis, polygon processing, interactive maps, and spatial statistics in one lightweight workflow.
</div>

Click the code cell below and press the **<code>▶</code> Run** button in the toolbar (or use `Shift + Enter`). The print messages will help you see exactly where we are in the workflow.

In [ ]:
print("Starting setup: importing geospatial libraries...")

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*keep_geom_type=True.*")

import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
import folium
import plotly.express as px
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, clear_output

# Problematic Imports
from libpysal import weights
import esda.moran as esda



pd.options.display.max_columns = 50
ox.settings.use_cache = True
ox.settings.log_console = False

print("Success! Core libraries are ready.")
print("We can now download real street, building, and green-space data.")

## 2. The Geographical Question

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Why this matters:</b> Cities are not just collections of roads and buildings. Their street layouts, block structure, and proximity to green space shape how places feel and function. In urban geoscience, these patterns help us explore accessibility, land-use intensity, and the spatial texture of neighbourhoods.
</div>

In this notebook, we will build a compact case study of central Edinburgh using **OpenStreetMap** data accessed through **OSMnx**.

We will connect:
- a real city study area
- street-network structure
- building-footprint morphology
- green-space proximity
- grid-based spatial aggregation
- hotspot-style interpretation

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Our plan:</b> Download a walkable street network and urban features, project everything into metres, calculate cell-level indicators, compare two analysis choices, then explore the results interactively.
</div>

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Exercise:</b> In the next code cell, look for comments beginning with <code># EDIT THIS VALUE</code>. Try changing the study radius or grid sizes after your first successful run, then re-run the notebook to see how scale changes the results.
</div>

In [ ]:
print("Defining our study area and downloading OpenStreetMap data...")

# EDIT THIS VALUE: central coordinate for a compact urban study area
study_centre = (55.9478, -3.1883)  # central Edinburgh

# EDIT THIS VALUE: radius in metres around the centre point
study_radius_m = 1400

# EDIT THIS VALUE: two grid sizes for a scale comparison
grid_sizes_m = [250, 500]

# EDIT THIS VALUE: access thresholds to compare
access_thresholds_m = [300, 600]

# EDIT THIS VALUE: how many cells to show in the explorer ranking table
top_n = 8

def osm_features_from_point(point, tags, dist):
    """Compatibility helper for different OSMnx versions."""
    if hasattr(ox, "features_from_point"):
        return ox.features_from_point(point, tags=tags, dist=dist)
    return ox.geometries_from_point(point, tags=tags, dist=dist)

network_type = "walk"

green_tags = {
    "leisure": ["park", "garden", "recreation_ground"],
    "landuse": ["grass", "recreation_ground", "village_green"],
    "natural": ["wood", "grassland"]
}

building_tags = {"building": True}

G = ox.graph_from_point(
    study_centre,
    dist=study_radius_m,
    network_type=network_type,
    simplify=True
)

greens_raw = osm_features_from_point(study_centre, green_tags, study_radius_m).reset_index(drop=True)
buildings_raw = osm_features_from_point(study_centre, building_tags, study_radius_m).reset_index(drop=True)

print(f"Walking network downloaded: {len(G.nodes):,} nodes and {len(G.edges):,} edges.")
print(f"Raw green-space features downloaded: {len(greens_raw):,}")
print(f"Raw building features downloaded: {len(buildings_raw):,}")

## 3. Inspecting the Urban Layers

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Clean the downloaded layers and inspect what they represent before we calculate any metrics.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Important concept:</b> Data from OpenStreetMap arrive in geographic coordinates (latitude and longitude). That is great for web maps, but not for measuring distance or area. For urban analysis, we should project the data into a metric coordinate system first.
</div>

Because our case study is in Scotland, we will use **British National Grid** (`EPSG:27700`). That lets us measure:
- length in metres
- area in square metres
- density per square kilometre
- distance to the nearest green space


In [ ]:
print("Projecting data to a metric CRS and cleaning the layers...")

projected_crs = "EPSG:27700"

G_proj = ox.project_graph(G, to_crs=projected_crs)
nodes, edges = ox.graph_to_gdfs(G_proj, nodes=True, edges=True)

greens = greens_raw.to_crs(projected_crs).copy()
buildings = buildings_raw.to_crs(projected_crs).copy()

greens = greens[greens.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
buildings = buildings[buildings.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()

greens["green_area_m2"] = greens.geometry.area
greens = greens[greens["green_area_m2"] >= 1000].copy()

buildings["footprint_area_m2"] = buildings.geometry.area
buildings = buildings[buildings["footprint_area_m2"] >= 50].copy()
buildings["perimeter_m"] = buildings.geometry.length
buildings["compactness"] = (
    4 * np.pi * buildings["footprint_area_m2"] / (buildings["perimeter_m"] ** 2)
).replace([np.inf, -np.inf], np.nan).clip(0, 1)

# Direct assignment avoids column collisions completely
nodes["street_count"] = nodes.index.map(ox.stats.count_streets_per_node(G_proj))

intersections = nodes[nodes["street_count"].fillna(0) >= 3].copy()

# FIX: Changed .unary_union to .union_all()
study_shape = edges.geometry.union_all().convex_hull.buffer(120)

study_area = gpd.GeoDataFrame(
    {"name": ["Study area"]},
    geometry=gpd.GeoSeries([study_shape], crs=projected_crs)
)

greens = gpd.clip(greens, study_area)
buildings = gpd.clip(buildings, study_area)
edges = gpd.clip(edges, study_area)
intersections = gpd.clip(intersections, study_area)

if greens.empty:
    raise ValueError("No green-space polygons were retained. Try increasing study_radius_m.")
if buildings.empty:
    raise ValueError("No building polygons were retained. Try increasing study_radius_m.")

print(f"Projected CRS: {projected_crs}")
print(f"Green spaces kept after cleaning: {len(greens):,}")
print(f"Buildings kept after cleaning: {len(buildings):,}")
print(f"Street intersections (3+ connecting streets): {len(intersections):,}")

fig, ax = plt.subplots(figsize=(9, 9))
study_area.boundary.plot(ax=ax, color="black", linewidth=1, alpha=0.6)
buildings.plot(ax=ax, color="#bdbdbd", linewidth=0, alpha=0.45)
greens.plot(ax=ax, color="#78c679", edgecolor="#238443", linewidth=0.4, alpha=0.8)
edges.plot(ax=ax, color="#636363", linewidth=0.7, alpha=0.7)
intersections.plot(ax=ax, color="#d7301f", markersize=8, alpha=0.9)
ax.set_title("Central Edinburgh: streets, buildings, and green spaces", fontsize=14)
ax.set_axis_off()
plt.show()

## 4. Building Urban Indicators on a Regular Grid

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>What is spatial aggregation?</b> We rarely analyse every street segment or building one-by-one when comparing neighbourhood structure. Instead, we aggregate information into consistent analysis units. Here, we will use a regular grid because it is lightweight, transparent, and easy to reproduce.
</div>

For each grid cell, we will calculate several interpretable metrics:

- **Intersection density**: a simple street-connectivity proxy
- **Street length density**: how much walkable street exists per square kilometre
- **Building coverage**: the share of land covered by building footprints
- **Mean building compactness**: a simple morphology measure derived from footprint shape
- **Nearest green-space distance**: the distance from the cell centroid to the nearest green polygon

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Compute these indicators for <b>two grid sizes</b> so we can see how aggregation scale changes the picture.
</div>

In [ ]:
print("Building analysis grids and calculating urban metrics...")

def make_grid(bounds, cell_size, crs):
    minx, miny, maxx, maxy = bounds
    xs = np.arange(minx, maxx + cell_size, cell_size)
    ys = np.arange(miny, maxy + cell_size, cell_size)
    wkts = []
    ids = []
    for i, x0 in enumerate(xs[:-1]):
        for j, y0 in enumerate(ys[:-1]):
            x1 = x0 + cell_size
            y1 = y0 + cell_size
            wkts.append(
                f"POLYGON(({x0} {y0}, {x1} {y0}, {x1} {y1}, {x0} {y1}, {x0} {y0}))"
            )
            ids.append(f"{cell_size}m_{i}_{j}")
    return gpd.GeoDataFrame(
        {"cell_id": ids},
        geometry=gpd.GeoSeries.from_wkt(wkts, crs=crs)
    )

def zscore(series):
    values = pd.Series(series).astype(float)
    std = values.std(ddof=0)
    if std == 0 or np.isnan(std):
        return pd.Series(np.zeros(len(values)), index=values.index)
    return (values - values.mean()) / std

def calculate_grid_metrics(cell_size):
    grid = make_grid(study_area.total_bounds, cell_size, projected_crs)
    grid = gpd.overlay(grid, study_area, how="intersection", keep_geom_type=False)
    grid = grid[grid.geometry.area >= 0.2 * (cell_size ** 2)].copy()
    grid["cell_area_m2"] = grid.geometry.area
    grid["cell_area_km2"] = grid["cell_area_m2"] / 1_000_000

    intersection_join = gpd.sjoin(
        intersections[["geometry"]],
        grid[["cell_id", "geometry"]],
        how="left",
        predicate="within"
    )
    intersection_counts = intersection_join.groupby("cell_id").size()
    grid["intersection_count"] = grid["cell_id"].map(intersection_counts).fillna(0).astype(int)
    grid["intersection_density_km2"] = grid["intersection_count"] / grid["cell_area_km2"]

    edge_clip = gpd.overlay(
        grid[["cell_id", "geometry"]],
        edges[["geometry"]],
        how="intersection",
        keep_geom_type=False
    )
    if edge_clip.empty:
        grid["street_length_m"] = 0.0
    else:
        edge_clip["street_m"] = edge_clip.geometry.length
        street_length = edge_clip.groupby("cell_id")["street_m"].sum()
        grid["street_length_m"] = grid["cell_id"].map(street_length).fillna(0)
    grid["street_length_density_km2"] = grid["street_length_m"] / grid["cell_area_km2"]

    building_clip = gpd.overlay(
        grid[["cell_id", "geometry"]],
        buildings[["geometry", "compactness"]],
        how="intersection",
        keep_geom_type=False
    )
    if building_clip.empty:
        grid["building_area_m2"] = 0.0
        grid["building_coverage_ratio"] = 0.0
        grid["building_coverage_pct"] = 0.0
        grid["mean_building_compactness"] = 0.0
    else:
        building_clip["building_area_m2"] = building_clip.geometry.area
        coverage = building_clip.groupby("cell_id")["building_area_m2"].sum()
        compactness_summary = building_clip.assign(
            weighted_compactness=building_clip["compactness"] * building_clip["building_area_m2"]
        ).groupby("cell_id").agg(
            compactness_area=("building_area_m2", "sum"),
            weighted_compactness=("weighted_compactness", "sum")
        )
        grid["building_area_m2"] = grid["cell_id"].map(coverage).fillna(0)
        grid["building_coverage_ratio"] = grid["building_area_m2"] / grid["cell_area_m2"]
        grid["building_coverage_pct"] = grid["building_coverage_ratio"] * 100
        grid["mean_building_compactness"] = (
            grid["cell_id"].map(compactness_summary["weighted_compactness"]).fillna(0)
            / grid["cell_id"].map(compactness_summary["compactness_area"]).replace(0, np.nan)
        ).fillna(0)

    centroids = grid[["cell_id", "geometry"]].copy()
    centroids["geometry"] = centroids.geometry.centroid
    nearest_green = gpd.sjoin_nearest(
        centroids,
        greens[["geometry"]],
        how="left",
        distance_col="nearest_green_m"
    )[["cell_id", "nearest_green_m"]].drop_duplicates(subset="cell_id")
    grid = grid.merge(nearest_green, on="cell_id", how="left")

    for threshold in access_thresholds_m:
        grid[f"green_access_{threshold}m"] = (grid["nearest_green_m"] <= threshold).astype(int)

    grid["street_green_balance"] = (
        zscore(grid["intersection_density_km2"])
        + zscore(grid["street_length_density_km2"])
        - zscore(grid["nearest_green_m"])
    ) / 3

    return grid.sort_values("cell_id").reset_index(drop=True)

results = {}
for cell_size in grid_sizes_m:
    print(f"  Processing {cell_size} m grid...")
    results[cell_size] = calculate_grid_metrics(cell_size)

metric_overview = pd.DataFrame({
    "Grid size (m)": [size for size in grid_sizes_m],
    "Cells kept": [len(results[size]) for size in grid_sizes_m],
    "Mean intersection density (/km²)": [round(results[size]["intersection_density_km2"].mean(), 1) for size in grid_sizes_m],
    "Median nearest green (m)": [round(results[size]["nearest_green_m"].median(), 1) for size in grid_sizes_m],
    "Mean building coverage (%)": [round(results[size]["building_coverage_pct"].mean(), 1) for size in grid_sizes_m]
})

print("Metric calculation complete.")
display(metric_overview)

## 5. Comparing Two Analysis Choices

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Comparison 1:</b> We compare <b>250 m</b> and <b>500 m</b> grid cells. This is a simple way to reveal a classic spatial-analysis issue: the same city can look different depending on the aggregation scale.
</div>
<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Comparison 2:</b> We compare <b>300 m</b> and <b>600 m</b> green-access thresholds. These are not universal or “correct” definitions. They are exploratory choices that help us see how sensitive the results are to the question we ask.
</div>

The charts below are designed to answer three practical questions:

1. How many cells count as "close" to green space under different thresholds?
2. How much does the distribution of nearest-green distance change when we change grid size?
3. Do dense street networks always line up with good green-space proximity?


In [ ]:
print("Creating summary charts for our two analysis choices...")

access_rows = []
distance_rows = []

for size, grid in results.items():
    label = f"{size} m grid"
    for threshold in access_thresholds_m:
        access_rows.append({
            "Grid size": label,
            "Threshold": f"{threshold} m",
            "Cells with a nearby green space (%)": 100 * grid[f"green_access_{threshold}m"].mean()
        })
    temp = grid[["nearest_green_m"]].copy()
    temp["Grid size"] = label
    distance_rows.append(temp)

access_df = pd.DataFrame(access_rows)
distance_df = pd.concat(distance_rows, ignore_index=True)

fig_access = px.bar(
    access_df,
    x="Threshold",
    y="Cells with a nearby green space (%)",
    color="Grid size",
    barmode="group",
    title="How much does the chosen access threshold matter?",
    color_discrete_sequence=["#3182bd", "#9ecae1"]
)
fig_access.update_layout(
    yaxis_title="Cells meeting the threshold (%)",
    xaxis_title="Nearest-green threshold"
)

fig_distance = px.histogram(
    distance_df,
    x="nearest_green_m",
    color="Grid size",
    nbins=18,
    barmode="overlay",
    opacity=0.65,
    title="Nearest green-space distance changes when we change aggregation scale",
    color_discrete_sequence=["#31a354", "#a1d99b"]
)
fig_distance.update_layout(
    xaxis_title="Distance from grid centroid to nearest green space (m)",
    yaxis_title="Number of cells"
)

scatter_grid = results[250].copy() if 250 in results else results[grid_sizes_m[0]].copy()
fig_scatter = px.scatter(
    scatter_grid,
    x="intersection_density_km2",
    y="nearest_green_m",
    color="building_coverage_pct",
    size="street_length_density_km2",
    hover_name="cell_id",
    title="A cell-level view: connectivity, green proximity, and building intensity",
    labels={
        "intersection_density_km2": "Intersection density (/km²)",
        "nearest_green_m": "Nearest green-space distance (m)",
        "building_coverage_pct": "Building coverage (%)",
        "street_length_density_km2": "Street length density"
    },
    color_continuous_scale="Viridis"
)
fig_scatter.update_layout(height=450)

fig_access.show()
fig_distance.show()
fig_scatter.show()

## 6. A Hotspot-Style Spatial Interpretation

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Move from simple mapping to a cautious spatial-clustering view. We will use Local Moran's I to see whether similar cells tend to cluster together.
</div>

Rather than running clustering on a single raw metric, we will use an **exploratory street-green balance score**. This score combines:
- higher street intersection density
- higher street length density
- shorter distance to green space

<div style="border:1px solid #f5c6cb; background:#f8d7da; color:#721c24; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Be careful:</b> This is a teaching-friendly composite indicator, not a definitive measure of liveability or environmental quality. It is a proxy designed to show how spatial statistics can be used responsibly with interpreted, not absolute, indicators.
</div>

In [ ]:
print("Running a hotspot-style clustering analysis on the exploratory street-green balance score...")

hotspot_grid_size = 250 if 250 in results else grid_sizes_m[0]
hotspot_grid = results[hotspot_grid_size].copy()

w = weights.Queen.from_dataframe(hotspot_grid)
w.transform = "R"

local_moran = esda.Moran_Local(hotspot_grid["street_green_balance"], w, permutations=99)
significant = local_moran.p_sim < 0.05

cluster_labels = np.where(
    significant & (local_moran.q == 1), "High-High",
    np.where(
        significant & (local_moran.q == 3), "Low-Low",
        np.where(
            significant & (local_moran.q == 2), "Low-High",
            np.where(significant & (local_moran.q == 4), "High-Low", "Not significant")
        )
    )
)

hotspot_grid["local_moran_I"] = local_moran.Is
hotspot_grid["cluster_type"] = cluster_labels

cluster_palette = {
    "High-High": "#d7191c",
    "Low-Low": "#2c7bb6",
    "Low-High": "#fdae61",
    "High-Low": "#abd9e9",
    "Not significant": "#dddddd"
}

fig, ax = plt.subplots(figsize=(9, 8))

# 1. Plot the dataframe layers (removed the 'label' argument from .plot() to avoid conflicting handlers)
for label, color in cluster_palette.items():
    hotspot_grid[hotspot_grid["cluster_type"] == label].plot(
        ax=ax,
        color=color,
        edgecolor="white",
        linewidth=0.6
    )
greens.plot(ax=ax, color="#78c679", edgecolor="none", alpha=0.35)

# 2. Fix: Create explicit proxy patch handles for the legend
legend_handles = [
    mpatches.Patch(facecolor=color, edgecolor="none", label=label)
    for label, color in cluster_palette.items()
]

ax.set_title(f"Local Moran clusters on the street-green balance score ({hotspot_grid_size} m grid)", fontsize=14)

# 3. Pass the custom proxy handles directly to the legend
ax.legend(handles=legend_handles, loc="lower left", frameon=True)

ax.set_axis_off()
plt.show()

cluster_summary = hotspot_grid["cluster_type"].value_counts().rename_axis("Cluster class").reset_index(name="Cell count")
display(cluster_summary)

print("Hotspot-style interpretation complete.")


## 7. Interactive Explorer

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Interactivity:</b> This is where the workflow becomes exploratory. You can switch between metrics and grid sizes to see how the same urban area looks under different analytical lenses.
</div>

Use the controls below to:
- change the mapped metric
- switch between the two grid sizes
- inspect the top-ranked cells for the chosen measure

Run the next cell, wait for the default map to appear, then try the dropdowns and slider.

In [ ]:
print("Building the interactive metric explorer...")

edges_ll = edges.to_crs(4326)
greens_ll = greens.to_crs(4326)

metric_options = {
    "Intersection density": ("intersection_density_km2", "High values mean many street junctions within the cell."),
    "Street length density": ("street_length_density_km2", "High values mean a dense walkable street fabric."),
    "Building coverage (%)": ("building_coverage_pct", "Higher values mean more land covered by building footprints."),
    "Mean building compactness": ("mean_building_compactness", "Higher values mean more geometrically compact building footprints."),
    "Nearest green-space distance (m)": ("nearest_green_m", "Lower values mean the cell centroid sits closer to green space."),
    "Exploratory street-green balance": ("street_green_balance", "A z-score blend of street connectivity and green proximity.")
}

palette = ["#f7fcf5", "#c7e9c0", "#74c476", "#31a354", "#006d2c"]

def add_quantile_colors(gdf, column, reverse=False):
    data = gdf[column].fillna(gdf[column].median())
    try:
        groups = pd.qcut(data.rank(method="first"), q=min(5, len(gdf)), labels=False, duplicates="drop")
    except ValueError:
        groups = pd.Series(np.zeros(len(gdf), dtype=int), index=gdf.index)
    colors = palette[::-1] if reverse else palette
    class_count = int(groups.max()) + 1 if len(groups) else 1
    use_colors = colors[:class_count]
    color_map = {i: use_colors[i] for i in range(class_count)}
    styled = gdf.copy()
    styled["fill_color"] = groups.map(color_map).fillna("#cccccc")
    return styled

def make_metric_map(grid, metric_name, map_title):
    reverse = metric_name in ["nearest_green_m"]
    styled = add_quantile_colors(grid.to_crs(4326), metric_name, reverse=reverse)
    m = folium.Map(location=study_centre, zoom_start=14, tiles="CartoDB positron")
    folium.GeoJson(
        greens_ll.to_json(),
        name="Green spaces",
        style_function=lambda x: {
            "fillColor": "#74c476",
            "color": "#238443",
            "weight": 0.6,
            "fillOpacity": 0.5
        }
    ).add_to(m)
    folium.GeoJson(
        edges_ll.to_json(),
        name="Street network",
        style_function=lambda x: {
            "color": "#666666",
            "weight": 1.0,
            "opacity": 0.5
        }
    ).add_to(m)
    folium.GeoJson(
        styled.to_json(),
        name=map_title,
        style_function=lambda feature: {
            "fillColor": feature["properties"]["fill_color"],
            "color": "#222222",
            "weight": 0.6,
            "fillOpacity": 0.65
        },
        tooltip=folium.GeoJsonTooltip(
            fields=[
                "cell_id",
                metric_name,
                "intersection_density_km2",
                "building_coverage_pct",
                "nearest_green_m"
            ],
            aliases=[
                "Cell:",
                f"{map_title}:",
                "Intersection density (/km²):",
                "Building coverage (%):",
                "Nearest green distance (m):"
            ],
            localize=True
        )
    ).add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    return m

def top_cell_table(grid, metric_name, top_n_value):
    ascending = metric_name == "nearest_green_m"
    
    # 1. Start with the metric_name, then add the other distinct base columns
    cols = [metric_name]
    for c in ["cell_id", "intersection_density_km2", "building_coverage_pct", "nearest_green_m"]:
        if c not in cols:
            cols.append(c)
            
    table = grid[cols].sort_values(metric_name, ascending=ascending).head(top_n_value).copy()
    
    # 2. Build the rename map, ignoring metric_name if it is already one of the keys
    rename_map = {
        "intersection_density_km2": "Intersection density (/km²)",
        "building_coverage_pct": "Building coverage (%)",
        "nearest_green_m": "Nearest green distance (m)"
    }
    # Ensure the chosen selected metric gets the primary title label
    rename_map[metric_name] = "Selected metric"
    
    return table.rename(columns=rename_map).round(2)

metric_widget = widgets.Dropdown(
    options=list(metric_options.keys()),
    value="Exploratory street-green balance",
    description="Metric:"
)

grid_widget = widgets.Dropdown(
    options=grid_sizes_m,
    value=250 if 250 in grid_sizes_m else grid_sizes_m[0],
    description="Grid size:"
)

topn_widget = widgets.IntSlider(
    value=top_n,
    min=3,
    max=12,
    step=1,
    description="Top cells:",
    continuous_update=False
)

def update_explorer(metric_label, grid_size, top_n_value):
    clear_output(wait=True)
    column, help_text = metric_options[metric_label]
    grid = results[grid_size].copy()
    print(f"Showing {metric_label} on a {grid_size} m grid...")
    print(help_text)
    display(make_metric_map(grid, column, metric_label))
    display(top_cell_table(grid, column, top_n_value))

controls = widgets.HBox([metric_widget, grid_widget, topn_widget])
interactive_view = widgets.interactive_output(
    update_explorer,
    {
        "metric_label": metric_widget,
        "grid_size": grid_widget,
        "top_n_value": topn_widget
    }
)

display(controls, interactive_view)

print("Interactive explorer ready! Try changing the metric or grid size.")

## 8. Responsible Interpretation and Limitations

<div style="border:1px solid #f5c6cb; background:#f8d7da; color:#721c24; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Important limitations:</b> These outputs are useful, but they are not the whole story. Spatial indicators are always shaped by definitions, data quality, and scale choices.
</div>

Here are the main cautions to keep in mind:

- **OpenStreetMap completeness varies.** Some buildings, paths, or green spaces may be missing, simplified, or tagged inconsistently.
- **Green space is a proxy, not a guarantee of access.** A polygon may be private, fenced, steep, poorly maintained, or difficult to enter from the street.
- **Our nearest-green calculation uses cell centroids.** That is lightweight and interpretable, but it is still a simplification of how people actually move.
- **The walk network is not the same as lived mobility.** Slope, crossings, steps, safety, and traffic conditions are not fully captured here.
- **Grid size matters.** The 250 m grid reveals more local variation; the 500 m grid smooths it. This is a classic scale effect and a reminder of the **modifiable areal unit problem (MAUP)**.
- **Composite scores should be handled carefully.** Our street-green balance measure is exploratory and should not be treated as a direct measure of quality, equity, or wellbeing.
- **Hotspot maps show local clustering, not causality.** A cluster can highlight a pattern worth investigating, but it does not explain why that pattern exists.

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Good scientific practice:</b> Treat these outputs as evidence for asking better spatial questions, not as final answers. In a fuller project, you might compare multiple cities, add deprivation or land-use data, use network travel time, or validate against field knowledge.
</div>

## Take-away

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    Congratulations! You have built a compact but powerful urban geoscience workflow. You downloaded real open data, handled projection correctly, created grid-based indicators, compared analysis choices, explored clustering, and used an interactive map to inspect the results. This is exactly the kind of reproducible spatial reasoning that scales well into more advanced GeoScience work.
</div>

### Suggested next steps

- Try a different city-centre coordinate or a larger study radius
- Add a second city and compare patterns between them
- Replace centroid distance with network distance for a more realistic accessibility analysis
- Explore additional morphology measures, such as block size or street orientation
- Join the grid to demographic or environmental data for a more complete urban analysis

If you make changes, re-run the notebook from top to bottom so each layer and metric updates consistently.

![Noteable license](https://raw.githubusercontent.com/YusufNik/Noteable/refs/heads/main/images/Noteable%20Notebook%20Footer.png)